# RAG手法比較ラボ（Google Colab）

このノートブックは `multilingual-e5-small` をColab GPUで実行します。スキャンPDFを扱う場合は、PaddleOCRも使います。実行前に、Colabの **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選んでください。

In [ ]:
!git clone https://github.com/wataru-i-0823/rag-method-benchmark.git /content/rag-method-benchmark
%cd /content/rag-method-benchmark/work
!pip -q install uv
!uv venv --python 3.11
!uv sync --extra colab --extra ocr

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/rag-method-benchmark/work/data')
for name in ('raw', 'processed', 'evaluation', 'mlruns', 'results'):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)

# 金融庁・日本銀行の公開ページを取得する。本文はDriveだけに保存され、Gitには追加されない。
!uv run python scripts/download_public_sources.py --raw-dir {DRIVE_ROOT / 'raw' / 'fsa-boj-public-information'} --output {DRIVE_ROOT / 'processed' / 'fsa_boj_public_information.jsonl'}
print(DRIVE_ROOT)

## PDF取込テスト

PDFをGoogle Driveの `raw/pdfs/` に置いてから次のセルを実行します。各ページはまずPyMuPDFで直接抽出され、文字が不足するページだけPaddleOCRで日本語OCRします。PDFと抽出結果はDrive内に残り、GitHubへは追加されません。

In [ ]:
PDF_DIR = DRIVE_ROOT / 'raw' / 'pdfs'
PDF_DIR.mkdir(parents=True, exist_ok=True)
# 例: Google Drive上のPDFをページ単位のJSONLコーパスへ変換する。
!uv run python scripts/ingest_pdfs.py --input-dir {PDF_DIR} --ocr-backend paddle --output {DRIVE_ROOT / 'processed' / 'pdf_corpus.jsonl'}
!head -n 1 {DRIVE_ROOT / 'processed' / 'pdf_corpus.jsonl'}

In [ ]:
# E5 + Chromaで公開情報を検索する。精度比較にはDriveのevaluation/に正解付き質問JSONLを置く。
!uv run python -m rag_lab inspect --profile colab \
  --corpus /content/drive/MyDrive/rag-method-benchmark/work/data/processed/fsa_boj_public_information.jsonl \
  --query '日本銀行の物価安定の目標は何ですか' --method chroma_e5

In [ ]:
# evaluation/qa.jsonlを用意した後に、以下を実行してMLflow結果をDriveへ保存する。
# !uv run python -m rag_lab evaluate --profile colab --corpus /content/drive/MyDrive/rag-method-benchmark/work/data/processed/fsa_boj_public_information.jsonl --qa /content/drive/MyDrive/rag-method-benchmark/work/data/evaluation/qa.jsonl --methods bm25,dense,hybrid,chroma_e5 --mlflow --tracking-uri file:///content/drive/MyDrive/rag-method-benchmark/work/data/mlruns --experiment fsa-boj-e5
# !cp -f results/* /content/drive/MyDrive/rag-method-benchmark/work/data/results/